#主成分分析により多重共線性の解消を目指す

In [1]:
#import
import pandas as pd
from sklearn.model_selection import KFold #K分割交差検証
from sklearn.model_selection import cross_validate  #K分割交差検証

from sklearn.linear_model import LinearRegression #回帰
from sklearn.preprocessing import PolynomialFeatures  #交互作用特徴量
from sklearn.linear_model import Ridge  #リッジ回帰

from statsmodels.stats.outliers_influence import variance_inflation_factor  #VIF
from sklearn.decomposition import PCA  #PCA

In [2]:
#データフレームの読み込み
'''
df1 VIFの高さを解消するために一部の列を消去する前のデータフレーム
df2 〃した後のデータフレーム

以降は基本的にdf1を用いて分析を行う。
sc_x,df_yはdf1を基に作成する。
ただし、必要があればdf1も利用する。
'''
df1 = pd.read_csv('datafiles/df1_all_col.csv')

sc_x = df1.drop(['SalePrice'], axis = 1)
df_y = pd.DataFrame(df1['SalePrice'])

In [3]:
#K分割交差検証のための準備
kf=KFold(n_splits=5, shuffle=True, random_state=0)

In [4]:
#モデルの作成
#リッジ回帰
model2 = Ridge(alpha = 100)
model2.fit(sc_x, df_y)

,alpha,100
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


In [ ]:
#THE_COL_NAME　は 'GrandFinish' のようにダミー変数化したい列の名
#ダミー変数化されたTHE_COL_NAME列に主成分分析を実施し、変数を減らしたsc_x1を返す関数（非破壊関数）
def change_to_PCA(THE_COL_NAME, sc_x, df_y):
    sc_x1 = sc_x.copy()
    df_y1 = df_y.copy()

    #THE_COL_NAME列をダミー変数化した列一覧をリスト化し、主成分分析により一列化
    THE_COL_NAME_cols = []
    for c in sc_x1.columns:
        if THE_COL_NAME in c:
            THE_COL_NAME_cols.append(c)
    print(THE_COL_NAME_cols)



    #累積寄与率の閾値を0.8として、n_componentsを設定
    #適切なPCAのための特徴量数の設定
    PCAmodel = PCA(whiten = True)
    THE_COL_NAME_df = pd.DataFrame()
    for c in THE_COL_NAME_cols:
        THE_COL_NAME_df = pd.concat([THE_COL_NAME_df, sc_x1[c]], axis = 1)
    PCAmodel.fit(THE_COL_NAME_df)
    the_col_name = PCAmodel.transform(sc_x1[THE_COL_NAME_cols])

    thred = 0.8
    final_num = 0
    ratio =PCAmodel.explained_variance_ratio_
    array = []
    for i in range(len(ratio)):
        ruiseki = sum(ratio[0:i+1])    #i+1個めの特徴量までの累積寄与率
        if ruiseki >= thred:   #i+1個めの特徴量において初めて累積寄与率がthredを超えるならば
            final_num = i + 1   #特徴量はi+1個必要である
            break
    print(f'PCAにより作成された特徴量数＝{final_num}')



    #最適な特徴量数で、主成分分析の実施
    PCAmodel = PCA(n_components = final_num, whiten = True)
    PCAmodel.fit(THE_COL_NAME_df)

    #主成分によるデータフレームをTHE_COL_NAME_PCA_dfとする
    THE_COL_NAME_PCA = PCAmodel.transform(THE_COL_NAME_df)
    THE_COL_NAME_PCA_df = pd.DataFrame(THE_COL_NAME_PCA)

    col_name = []
    for i in range(final_num):
        name = 'col_name_' + str(i)
        col_name.append(name)
    THE_COL_NAME_PCA_df.columns = col_name



    #主成分分析により作成した列を、実施前の列と置き換える
    for c in sc_x1[THE_COL_NAME_cols]:
        sc_x1 = sc_x1.drop([c], axis = 1)
    sc_x1 = pd.concat([sc_x1, THE_COL_NAME_PCA_df], axis = 1)



        #主成分分析により作成した列を、実施前の列と置き換える
    for c in sc_x1[THE_COL_NAME_cols]:
        sc_x1 = sc_x1.drop([c], axis = 1)
    sc_x1 = pd.concat([sc_x1, THE_COL_NAME_PCA_df], axis = 1)



    return sc_x1

In [ ]:
change_to_PCA('GrandFinish', sc_x, df_y)

[]


ValueError: at least one array or dtype is required

In [ ]:
'''
4_analysis_1 でVIFがinfであった列
BsmtFinSF1, BsmtFinSF2, BsmtUnfSF, TotalBsmtSF, 1stFlrSF, 2ndFlrSF, LowQualFinSF, GrLivArea, Exterior2nd_CBlock, Exterior1st_CBlock
'''

'\n4_analysis_1 でVIFがinfであった列\nBsmtFinSF1, BsmtFinSF2, BsmtUnfSF, TotalBsmtSF, 1stFlrSF, 2ndFlrSF, LowQualFinSF, GrLivArea, Exterior2nd_CBlock, Exterior1st_CBlock\n'